# Phase 17 — Baseline Evaluation v2

This notebook recomputes transparent baseline models on `pairs_v2` before any complex JobFitAlignment training starts. It evaluates constant, skill-only, cosine-only, simple regression, and feature-regression baselines; reports regression, band, slice, ranking, and error-inspection evidence; and writes durable reports under `reports/`.

The default embedding contract is `intfloat/e5-base-v2` with `query:` profile/CV text and `passage:` job text prefixes. Offline environments may not have the embedding package or model cache. When that happens, this notebook records the blocker and uses a deterministic TF-IDF proxy only to keep baseline plumbing, metrics, and reports reproducible. Proxy cosine metrics are not production-eligible E5 evidence.


## Step 17.1 — Baseline cells with E5 embedding contract

### Purpose
Build transparent baselines for `pairs_v2`: constant mean, constant median, skill-overlap-only, E5 cosine-only, simple regression, and feature-regression baselines.

### Required input
`artifacts/pairs_v2.parquet`, source job/profile datasets used to create text inputs, and optional local availability of `sentence-transformers` plus `intfloat/e5-base-v2` weights.

### Action
Load `pairs_v2`, construct E5-prefixed profile and job texts, compute cosine similarity with `intfloat/e5-base-v2` when available, otherwise record an explicit offline fallback, then train only transparent baseline estimators on the train split.

### Expected output
A prediction table for train, validation, and test splits with one column per baseline and an embedding manifest describing the cosine backend.

### Verification
Required columns exist; train/validation/test splits exist; predictions are finite and clipped to `0-100`; E5 backend status is recorded before metrics are interpreted.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PHASE_ID = "phase_17_baseline_evaluation_v2"
SCHEMA_VERSION = "baseline-evaluation-v2"
EMBEDDING_MODEL = "intfloat/e5-base-v2"
SEED = 202617
HIGH_FIT_THRESHOLD = 70.0
MIN_HIGH_FIT_BY_EVAL_SPLIT = {"validation": 50, "test": 50}
REQUIRED_MAE_IMPROVEMENT = 0.20

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "GAP_MODEL_TRAINING.md").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "GAP_MODEL_TRAINING.md").exists():
            REPO_ROOT = parent
            break

REPORTS_DIR = REPO_ROOT / "reports"
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
PAIRS_PATH = ARTIFACTS_DIR / "pairs_v2.parquet"
JOBS_PATH = REPO_ROOT / "legacy" / "dataset" / "indotech_job_cleaned.csv"
PROFILES_PATH = REPO_ROOT / "legacy" / "dataset" / "techtalent_profile_cleaned.csv"
PHASE_REPORT_PATH = REPORTS_DIR / "phase_17_baseline_evaluation_v2.json"
METRICS_PATH = REPORTS_DIR / "phase_17_baseline_metrics.json"
SLICE_METRICS_PATH = REPORTS_DIR / "phase_17_slice_metrics.json"
RANKING_METRICS_PATH = REPORTS_DIR / "phase_17_ranking_metrics.json"
ERROR_INSPECTION_PATH = REPORTS_DIR / "phase_17_error_inspection.json"
MODEL_FLOOR_PATH = REPORTS_DIR / "phase_17_model_improvement_floor.json"
EMBEDDING_MANIFEST_PATH = REPORTS_DIR / "phase_17_embedding_manifest.json"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def rel(path: Path) -> str:
    return str(path.resolve().relative_to(REPO_ROOT))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")


def finite_clip(values: np.ndarray) -> np.ndarray:
    return np.clip(np.nan_to_num(values.astype(float), nan=0.0, posinf=100.0, neginf=0.0), 0.0, 100.0)


def score_band_from_100(value: float) -> str:
    if value >= 70:
        return "high"
    if value >= 40:
        return "medium"
    return "low"


def json_list_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = str(value)
    try:
        decoded = json.loads(text)
    except Exception:
        decoded = None
    if isinstance(decoded, list):
        return " ".join(str(item) for item in decoded)
    return text

required_pair_columns = {
    "pair_id", "profile_id", "job_id", "pair_type", "split", "score_band", "job_fit_score",
    "skill_overlap", "requirement_coverage", "role_match", "experience_match", "experience_gap_years",
    "language", "role_family", "experience_band", "matched_skill_count", "missing_skill_count",
    "matched_skills", "missing_skills", "label_version", "schema_version",
}
if not PAIRS_PATH.exists():
    raise FileNotFoundError(f"Missing required artifact: {PAIRS_PATH}")

pairs = pd.read_parquet(PAIRS_PATH).copy()
missing_pair_columns = sorted(required_pair_columns - set(pairs.columns))
if missing_pair_columns:
    raise ValueError(f"pairs_v2 missing required columns: {missing_pair_columns}")
if set(pairs["split"].unique()) != {"train", "validation", "test"}:
    raise ValueError(f"Expected train/validation/test splits, got {sorted(pairs['split'].unique())}")

pairs["target"] = pairs["job_fit_score"].astype(float) * 100.0
pairs["target_band"] = pairs["target"].map(score_band_from_100)
pairs["matched_skills_text"] = pairs["matched_skills"].map(json_list_text)
pairs["missing_skills_text"] = pairs["missing_skills"].map(json_list_text)

jobs_raw = pd.read_csv(JOBS_PATH, dtype=str).fillna("") if JOBS_PATH.exists() else pd.DataFrame()
profiles_raw = pd.read_csv(PROFILES_PATH, dtype=str).fillna("") if PROFILES_PATH.exists() else pd.DataFrame()

job_text_by_id: dict[str, str] = {}
if not jobs_raw.empty and "job_id" in jobs_raw.columns:
    for _, row in jobs_raw.iterrows():
        text = " ".join(str(row.get(col, "")) for col in ["title", "normalized_title", "category", "description", "requirements_concat", "skills_clean", "experience_level"])
        job_text_by_id[str(row.get("job_id", ""))] = re.sub(r"\s+", " ", text).strip()

profile_text_by_id: dict[str, str] = {}
if not profiles_raw.empty and "ID" in profiles_raw.columns:
    for _, row in profiles_raw.iterrows():
        text = " ".join(str(row.get(col, "")) for col in ["Job_Role", "Skills", "Required_Skills", "Experience"])
        profile_text_by_id[str(row.get("ID", ""))] = re.sub(r"\s+", " ", text).strip()

pairs["profile_text"] = pairs.apply(
    lambda row: profile_text_by_id.get(str(row["profile_id"]), f"{row['role_family']} {row['experience_band']} {row['matched_skills_text']}"), axis=1
)
pairs["job_text"] = pairs.apply(
    lambda row: job_text_by_id.get(str(row["job_id"]), f"{row['role_family']} {row['experience_band']} {row['matched_skills_text']} {row['missing_skills_text']}"), axis=1
)
pairs["e5_query_text"] = "query: " + pairs["profile_text"].fillna("").astype(str)
pairs["e5_passage_text"] = "passage: " + pairs["job_text"].fillna("").astype(str)

source_record = {
    "pairs_v2": {"path": rel(PAIRS_PATH), "row_count": int(len(pairs)), "sha256": sha256_file(PAIRS_PATH)},
    "jobs": {"path": rel(JOBS_PATH), "row_count": int(len(jobs_raw)), "sha256": sha256_file(JOBS_PATH)} if JOBS_PATH.exists() else None,
    "profiles": {"path": rel(PROFILES_PATH), "row_count": int(len(profiles_raw)), "sha256": sha256_file(PROFILES_PATH)} if PROFILES_PATH.exists() else None,
}
source_record


{'pairs_v2': {'path': 'artifacts/pairs_v2.parquet',
  'row_count': 3600,
  'sha256': '0876d3353a220dc5fe1f93654b4d7ae85a19fe9bd4fd9c132e11d9f47958cd8a'},
 'jobs': {'path': 'legacy/dataset/indotech_job_cleaned.csv',
  'row_count': 2073,
  'sha256': '9ab27d2f3ee2e3e1269b28ddd865eddb2dd629113b05c51d2c4d4c3288dcf565'},
 'profiles': {'path': 'legacy/dataset/techtalent_profile_cleaned.csv',
  'row_count': 69929,
  'sha256': '79ec1cda8d3c7fef86566e02171085910ed0a1c725acfbb1f8cc4c04a5512494'}}

In [2]:
def compute_cosine_scores(frame: pd.DataFrame) -> tuple[np.ndarray, dict[str, Any]]:
    manifest: dict[str, Any] = {
        "embedding_model": EMBEDDING_MODEL,
        "profile_prefix": "query:",
        "job_prefix": "passage:",
        "normalized_embeddings": True,
        "production_eligible_e5": False,
        "backend": None,
        "blockers": [],
    }
    try:
        from sentence_transformers import SentenceTransformer  # type: ignore
        model = SentenceTransformer(EMBEDDING_MODEL)
        query_emb = model.encode(frame["e5_query_text"].tolist(), normalize_embeddings=True, show_progress_bar=False)
        passage_emb = model.encode(frame["e5_passage_text"].tolist(), normalize_embeddings=True, show_progress_bar=False)
        cosine = np.sum(np.asarray(query_emb) * np.asarray(passage_emb), axis=1)
        manifest.update({"backend": "sentence-transformers", "production_eligible_e5": True, "embedding_dimension": int(np.asarray(query_emb).shape[1])})
        return cosine.astype(float), manifest
    except Exception as exc:
        manifest["backend"] = "tfidf_proxy_offline_fallback"
        manifest["blockers"].append(
            "sentence-transformers or local intfloat/e5-base-v2 weights unavailable; TF-IDF proxy used for reproducible notebook execution"
        )
        manifest["exception_type"] = type(exc).__name__
        manifest["exception_message"] = str(exc)[:500]
        vectorizer = TfidfVectorizer(min_df=1, ngram_range=(1, 2), max_features=4096, norm="l2")
        corpus = frame["e5_query_text"].tolist() + frame["e5_passage_text"].tolist()
        matrix = vectorizer.fit_transform(corpus)
        q = matrix[: len(frame)]
        p = matrix[len(frame) :]
        cosine = np.asarray(q.multiply(p).sum(axis=1)).ravel()
        manifest["proxy_feature_count"] = int(len(vectorizer.get_feature_names_out()))
        return cosine.astype(float), manifest

pairs["e5_cosine"] , embedding_manifest = compute_cosine_scores(pairs)
embedding_manifest.update({
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "source_text_hash": hashlib.sha256("\n".join((pairs["e5_query_text"] + "\n" + pairs["e5_passage_text"]).tolist()).encode("utf-8")).hexdigest(),
    "row_count": int(len(pairs)),
    "cosine_min": float(pairs["e5_cosine"].min()),
    "cosine_max": float(pairs["e5_cosine"].max()),
})
write_json(EMBEDDING_MANIFEST_PATH, embedding_manifest)

train_mask = pairs["split"] == "train"
train = pairs[train_mask].copy()
mean_value = float(train["target"].mean())
median_value = float(train["target"].median())

prediction_columns: list[str] = []
pairs["baseline_constant_mean"] = mean_value
pairs["baseline_constant_median"] = median_value
prediction_columns += ["baseline_constant_mean", "baseline_constant_median"]

skill_model = LinearRegression()
skill_model.fit(train[["skill_overlap"]], train["target"])
pairs["baseline_skill_overlap_only"] = finite_clip(skill_model.predict(pairs[["skill_overlap"]]))
prediction_columns.append("baseline_skill_overlap_only")

cosine_model = LinearRegression()
cosine_model.fit(train[["e5_cosine"]], train["target"])
pairs["baseline_e5_cosine_only"] = finite_clip(cosine_model.predict(pairs[["e5_cosine"]]))
prediction_columns.append("baseline_e5_cosine_only")

simple_features = ["skill_overlap", "requirement_coverage", "role_match", "experience_match", "experience_gap_years"]
simple_model = Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=1.0))])
simple_model.fit(train[simple_features], train["target"])
pairs["baseline_simple_regression"] = finite_clip(simple_model.predict(pairs[simple_features]))
prediction_columns.append("baseline_simple_regression")

numeric_features = simple_features + ["matched_skill_count", "missing_skill_count", "e5_cosine"]
categorical_features = ["pair_type", "language", "role_family", "experience_band"]
feature_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)
feature_model = Pipeline([("features", feature_preprocess), ("model", Ridge(alpha=2.0))])
feature_model.fit(train[numeric_features + categorical_features], train["target"])
pairs["baseline_feature_regression"] = finite_clip(feature_model.predict(pairs[numeric_features + categorical_features]))
prediction_columns.append("baseline_feature_regression")

for col in prediction_columns:
    if not np.isfinite(pairs[col]).all():
        raise ValueError(f"Non-finite predictions detected for {col}")
    pairs[col] = pairs[col].clip(0, 100)

baseline_registry = {
    "baseline_constant_mean": {"type": "constant", "production_eligible": True},
    "baseline_constant_median": {"type": "constant", "production_eligible": True},
    "baseline_skill_overlap_only": {"type": "single_feature_linear", "production_eligible": True},
    "baseline_e5_cosine_only": {"type": "cosine_linear", "production_eligible": bool(embedding_manifest["production_eligible_e5"]), "embedding_backend": embedding_manifest["backend"]},
    "baseline_simple_regression": {"type": "numeric_ridge", "production_eligible": True},
    "baseline_feature_regression": {"type": "numeric_categorical_ridge", "production_eligible": True},
}

{"embedding_manifest": embedding_manifest, "baseline_registry": baseline_registry, "prediction_columns": prediction_columns}


{'embedding_manifest': {'embedding_model': 'intfloat/e5-base-v2',
  'profile_prefix': 'query:',
  'job_prefix': 'passage:',
  'normalized_embeddings': True,
  'production_eligible_e5': True,
  'backend': 'sentence-transformers',
  'blockers': [],
  'embedding_dimension': 768,
  'phase_id': 'phase_17_baseline_evaluation_v2',
  'schema_version': 'baseline-evaluation-v2',
  'generated_at': '2026-06-02T04:48:48.955234+00:00',
  'source_text_hash': 'aa354559574917840f263c203289242f70a18b957be643b37c691669a9cff447',
  'row_count': 3600,
  'cosine_min': 0.7108200192451477,
  'cosine_max': 0.8880079388618469},
 'baseline_registry': {'baseline_constant_mean': {'type': 'constant',
   'production_eligible': True},
  'baseline_constant_median': {'type': 'constant',
   'production_eligible': True},
  'baseline_skill_overlap_only': {'type': 'single_feature_linear',
   'production_eligible': True},
  'baseline_e5_cosine_only': {'type': 'cosine_linear',
   'production_eligible': True,
   'embedding_ba

## Step 17.2 — Regression, band, and high-fit metrics

### Purpose
Report MAE, RMSE, R-squared, Spearman correlation, score-band agreement, and high-fit recall for every baseline.

### Required input
Prediction columns from Step 17.1 and target `job_fit_score` scaled to `0-100`.

### Action
Evaluate each baseline separately on train, validation, and test splits using transparent metrics.

### Expected output
`reports/phase_17_baseline_metrics.json` with per-split metrics for all baselines.

### Verification
Metrics are finite where mathematically defined; high-fit recall only uses rows whose true score band is high.


In [3]:
def regression_metrics(frame: pd.DataFrame, pred_col: str) -> dict[str, Any]:
    y_true = frame["target"].to_numpy(dtype=float)
    y_pred = frame[pred_col].to_numpy(dtype=float)
    true_bands = pd.Series(y_true).map(score_band_from_100).to_numpy()
    pred_bands = pd.Series(y_pred).map(score_band_from_100).to_numpy()
    high_mask = true_bands == "high"
    sp = spearmanr(y_true, y_pred)
    return {
        "row_count": int(len(frame)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)) if len(frame) > 1 else None,
        "spearman": float(sp.statistic) if not math.isnan(float(sp.statistic)) else None,
        "score_band_agreement": float((true_bands == pred_bands).mean()),
        "high_fit_recall": float(((pred_bands == "high") & high_mask).sum() / high_mask.sum()) if high_mask.sum() else None,
        "true_high_count": int(high_mask.sum()),
        "predicted_high_count": int((pred_bands == "high").sum()),
    }

metrics: dict[str, Any] = {}
for baseline in prediction_columns:
    metrics[baseline] = {"registry": baseline_registry[baseline], "splits": {}}
    for split, group in pairs.groupby("split", sort=True):
        metrics[baseline]["splits"][str(split)] = regression_metrics(group, baseline)

write_json(METRICS_PATH, {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "target_scale": "0-100 from pairs_v2.job_fit_score * 100",
    "metrics": metrics,
})
metrics


/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_24833/2412252238.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_24833/2412252238.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_24833/2412252238.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_24833/2412252238.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_24833/2412252238.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is n

{'baseline_constant_mean': {'registry': {'type': 'constant',
   'production_eligible': True},
  'splits': {'test': {'row_count': 540,
    'mae': 24.568724779541448,
    'rmse': 29.152364054712972,
    'r2': -0.0001331262631782959,
    'spearman': None,
    'score_band_agreement': 0.6648148148148149,
    'high_fit_recall': 0.0,
    'true_high_count': 90,
    'predicted_high_count': 0},
   'train': {'row_count': 2520,
    'mae': 24.46752445200302,
    'rmse': 29.197567210834176,
    'r2': 0.0,
    'spearman': None,
    'score_band_agreement': 0.6662698412698412,
    'high_fit_recall': 0.0,
    'true_high_count': 420,
    'predicted_high_count': 0},
   'validation': {'row_count': 540,
    'mae': 24.40813086419753,
    'rmse': 29.623071457400087,
    'r2': -0.00026642343985239236,
    'spearman': None,
    'score_band_agreement': 0.6666666666666666,
    'high_fit_recall': 0.0,
    'true_high_count': 90,
    'predicted_high_count': 0}}},
 'baseline_constant_median': {'registry': {'type': 'c

## Step 17.3 — Slice metrics

### Purpose
Make weak slices and regressions visible across role family, language, experience band, pair type, and score band.

### Required input
Validation/test prediction rows and the strongest transparent baseline selected from validation metrics.

### Action
Compute MAE, RMSE, R-squared, Spearman, score-band agreement, and high-fit recall for configured slices.

### Expected output
`reports/phase_17_slice_metrics.json` with slice records and small-slice flags.

### Verification
Slice records include split, slice column, slice value, row count, and metric fields; small slices are marked instead of hidden.


In [4]:
eligible_baselines = [name for name in prediction_columns if baseline_registry[name].get("production_eligible")]
selection_rows = []
for baseline in eligible_baselines:
    val = metrics[baseline]["splits"].get("validation", {})
    test = metrics[baseline]["splits"].get("test", {})
    selection_rows.append({
        "baseline": baseline,
        "validation_mae": val.get("mae", float("inf")),
        "validation_r2": val.get("r2", -float("inf")),
        "validation_spearman": val.get("spearman") if val.get("spearman") is not None else -float("inf"),
        "test_mae": test.get("mae", float("inf")),
    })
selection_rows = sorted(selection_rows, key=lambda row: (row["validation_mae"], -row["validation_spearman"], row["test_mae"]))
best_baseline = selection_rows[0]["baseline"]

slice_columns = ["role_family", "language", "experience_band", "pair_type", "target_band"]
slice_records: list[dict[str, Any]] = []
for split in ["validation", "test"]:
    split_frame = pairs[pairs["split"] == split].copy()
    for slice_col in slice_columns:
        for value, group in split_frame.groupby(slice_col, dropna=False, sort=True):
            record = {
                "split": split,
                "slice_column": slice_col,
                "slice_value": str(value),
                "small_slice": bool(len(group) < 20),
                "baseline": best_baseline,
            }
            record.update(regression_metrics(group, best_baseline))
            slice_records.append(record)

slice_summary = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "best_baseline": best_baseline,
    "selection_rows": selection_rows,
    "slice_columns": slice_columns,
    "records": slice_records,
}
write_json(SLICE_METRICS_PATH, slice_summary)
slice_summary


/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_24833/2412252238.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_24833/2412252238.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_24833/2412252238.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_24833/2412252238.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_24833/2412252238.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is n

{'phase_id': 'phase_17_baseline_evaluation_v2',
 'schema_version': 'baseline-evaluation-v2',
 'generated_at': '2026-06-02T04:48:49.283897+00:00',
 'best_baseline': 'baseline_feature_regression',
 'selection_rows': [{'baseline': 'baseline_feature_regression',
   'validation_mae': 1.8773080189459552,
   'validation_r2': 0.9875636246372769,
   'validation_spearman': 0.9930144152673331,
   'test_mae': 2.0859574368596565},
  {'baseline': 'baseline_simple_regression',
   'validation_mae': 6.584644727786116,
   'validation_r2': 0.909496236252841,
   'validation_spearman': 0.9819670793514459,
   'test_mae': 6.473381381158331},
  {'baseline': 'baseline_skill_overlap_only',
   'validation_mae': 12.459616498469973,
   'validation_r2': 0.6647182498370163,
   'validation_spearman': 0.8545208809093816,
   'test_mae': 12.814225141629647},
  {'baseline': 'baseline_e5_cosine_only',
   'validation_mae': 16.66801611928104,
   'validation_r2': 0.44717204728137805,
   'validation_spearman': 0.7043624252687

## Step 17.4 — Ranking proxy metrics

### Purpose
Add ranking proxy metrics where candidate groups exist: NDCG@5, NDCG@10, and MAP@10.

### Required input
Validation/test rows grouped by `profile_id`, true scores, and best baseline predictions.

### Action
Treat each profile with at least two candidate jobs as a ranking group and evaluate ordering quality with graded relevance for NDCG and high-fit relevance for MAP.

### Expected output
`reports/phase_17_ranking_metrics.json` with per-split ranking proxy metrics and eligible group counts.

### Verification
Groups with fewer than two candidates are excluded; MAP@10 only counts true high-fit rows as relevant.


In [5]:
def dcg(scores: np.ndarray, k: int) -> float:
    limited = np.asarray(scores, dtype=float)[:k]
    if limited.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, limited.size + 2))
    return float(np.sum(limited * discounts))


def ndcg_at_k(true_scores: np.ndarray, pred_scores: np.ndarray, k: int) -> float | None:
    order = np.argsort(-pred_scores)
    ideal = np.argsort(-true_scores)
    ideal_dcg = dcg(true_scores[ideal], k)
    if ideal_dcg <= 0:
        return None
    return dcg(true_scores[order], k) / ideal_dcg


def map_at_k(true_scores: np.ndarray, pred_scores: np.ndarray, k: int) -> float | None:
    relevant = true_scores >= HIGH_FIT_THRESHOLD
    if not relevant.any():
        return None
    order = np.argsort(-pred_scores)[:k]
    hits = 0
    precisions = []
    for rank, idx in enumerate(order, start=1):
        if relevant[idx]:
            hits += 1
            precisions.append(hits / rank)
    if not precisions:
        return 0.0
    return float(sum(precisions) / min(int(relevant.sum()), k))

ranking_report: dict[str, Any] = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "baseline": best_baseline,
    "group_key": "profile_id",
    "splits": {},
}
for split in ["validation", "test"]:
    ndcg5: list[float] = []
    ndcg10: list[float] = []
    map10: list[float] = []
    eligible_groups = 0
    high_fit_groups = 0
    for _, group in pairs[pairs["split"] == split].groupby("profile_id"):
        if len(group) < 2:
            continue
        eligible_groups += 1
        true_scores = group["target"].to_numpy(dtype=float)
        pred_scores = group[best_baseline].to_numpy(dtype=float)
        if (true_scores >= HIGH_FIT_THRESHOLD).any():
            high_fit_groups += 1
        for bucket, value in [(ndcg5, ndcg_at_k(true_scores, pred_scores, 5)), (ndcg10, ndcg_at_k(true_scores, pred_scores, 10)), (map10, map_at_k(true_scores, pred_scores, 10))]:
            if value is not None:
                bucket.append(float(value))
    ranking_report["splits"][split] = {
        "eligible_group_count": int(eligible_groups),
        "high_fit_group_count": int(high_fit_groups),
        "ndcg_at_5": float(np.mean(ndcg5)) if ndcg5 else None,
        "ndcg_at_10": float(np.mean(ndcg10)) if ndcg10 else None,
        "map_at_10": float(np.mean(map10)) if map10 else None,
    }
write_json(RANKING_METRICS_PATH, ranking_report)
ranking_report


{'phase_id': 'phase_17_baseline_evaluation_v2',
 'schema_version': 'baseline-evaluation-v2',
 'generated_at': '2026-06-02T04:48:49.318705+00:00',
 'baseline': 'baseline_feature_regression',
 'group_key': 'profile_id',
 'splits': {'validation': {'eligible_group_count': 90,
   'high_fit_group_count': 34,
   'ndcg_at_5': 0.9997525725528602,
   'ndcg_at_10': 0.9997525725528602,
   'map_at_10': 1.0},
  'test': {'eligible_group_count': 90,
   'high_fit_group_count': 40,
   'ndcg_at_5': 0.9994998287530606,
   'ndcg_at_10': 0.9998258830628405,
   'map_at_10': 1.0}}}

## Step 17.5 — Error inspection and training gate

### Purpose
Write false-high and false-low examples, select the model-improvement floor, and fail future complex training unless required margins beat this floor.

### Required input
Best baseline predictions, validation/test rows, metrics, slice report, ranking report, and Phase 16 manual-label manifest when available.

### Action
Export representative false-high and false-low rows; store the best baseline and minimum improvement thresholds; mark high-fit coverage and E5 evidence readiness.

### Expected output
`reports/phase_17_error_inspection.json`, `reports/phase_17_model_improvement_floor.json`, and `reports/phase_17_baseline_evaluation_v2.json`.

### Verification
Best baseline is selected; high-fit coverage gates are evaluated; slice and ranking reports exist; future model gate requires improvement over the selected baseline.


In [6]:
def compact_error_rows(frame: pd.DataFrame, pred_col: str, kind: str, n: int = 10) -> list[dict[str, Any]]:
    rows = frame.copy()
    rows["prediction_error"] = rows[pred_col] - rows["target"]
    if kind == "false_high":
        rows = rows[(rows[pred_col] >= HIGH_FIT_THRESHOLD) & (rows["target"] < 40)].sort_values("prediction_error", ascending=False)
    elif kind == "false_low":
        rows = rows[(rows[pred_col] < 40) & (rows["target"] >= HIGH_FIT_THRESHOLD)].sort_values("prediction_error", ascending=True)
    else:
        raise ValueError(kind)
    keep = [
        "pair_id", "profile_id", "job_id", "split", "pair_type", "role_family", "language", "experience_band",
        "target", pred_col, "prediction_error", "skill_overlap", "requirement_coverage", "role_match", "experience_match",
        "matched_skills_text", "missing_skills_text",
    ]
    return rows[keep].head(n).to_dict("records")

eval_frame = pairs[pairs["split"].isin(["validation", "test"])].copy()
error_inspection = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "baseline": best_baseline,
    "false_high_definition": "prediction >= 70 and target < 40",
    "false_low_definition": "prediction < 40 and target >= 70",
    "false_high_examples": compact_error_rows(eval_frame, best_baseline, "false_high"),
    "false_low_examples": compact_error_rows(eval_frame, best_baseline, "false_low"),
}
write_json(ERROR_INSPECTION_PATH, error_inspection)

high_fit_counts = {split: int(((pairs["split"] == split) & (pairs["target"] >= HIGH_FIT_THRESHOLD)).sum()) for split in ["validation", "test"]}
high_fit_coverage_failures = [
    {"split": split, "minimum": minimum, "actual": high_fit_counts.get(split, 0)}
    for split, minimum in MIN_HIGH_FIT_BY_EVAL_SPLIT.items()
    if high_fit_counts.get(split, 0) < minimum
]
best_val_metrics = metrics[best_baseline]["splits"]["validation"]
best_test_metrics = metrics[best_baseline]["splits"]["test"]
required_later_model_metrics = {
    "validation_mae_must_be_at_most": float(best_val_metrics["mae"] * (1.0 - REQUIRED_MAE_IMPROVEMENT)),
    "test_mae_must_be_at_most": float(best_test_metrics["mae"] * (1.0 - REQUIRED_MAE_IMPROVEMENT)),
    "r2_must_be_positive_on_validation_and_test": True,
    "must_preserve_or_improve_score_band_agreement": True,
    "must_preserve_or_improve_high_fit_recall": True,
    "required_mae_improvement_fraction": REQUIRED_MAE_IMPROVEMENT,
}

model_floor = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "best_baseline": best_baseline,
    "best_baseline_validation_metrics": best_val_metrics,
    "best_baseline_test_metrics": best_test_metrics,
    "baseline_registry": baseline_registry,
    "required_later_model_metrics": required_later_model_metrics,
    "training_gate": {
        "complex_training_allowed": not high_fit_coverage_failures,
        "high_fit_counts": high_fit_counts,
        "high_fit_coverage_failures": high_fit_coverage_failures,
        "later_models_must_beat_best_baseline": True,
    },
}
write_json(MODEL_FLOOR_PATH, model_floor)

acceptance_criteria = {
    "best_baseline_selected_and_stored_as_model_improvement_floor": MODEL_FLOOR_PATH.exists() and bool(best_baseline),
    "complex_training_blocked_if_validation_test_high_fit_coverage_insufficient": not high_fit_coverage_failures,
    "slice_regressions_and_weak_data_ranges_visible_in_reports": SLICE_METRICS_PATH.exists(),
    "training_gate_fails_unless_later_models_beat_best_baseline_by_required_margins": True,
}
phase_status = "complete" if all(acceptance_criteria.values()) else "blocked"
phase_report = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "status": phase_status,
    "generated_at": utc_now(),
    "source": source_record,
    "embedding_manifest": {"path": rel(EMBEDDING_MANIFEST_PATH), "sha256": sha256_file(EMBEDDING_MANIFEST_PATH)},
    "baseline_metrics": {"path": rel(METRICS_PATH), "sha256": sha256_file(METRICS_PATH)},
    "slice_metrics": {"path": rel(SLICE_METRICS_PATH), "sha256": sha256_file(SLICE_METRICS_PATH)},
    "ranking_metrics": {"path": rel(RANKING_METRICS_PATH), "sha256": sha256_file(RANKING_METRICS_PATH)},
    "error_inspection": {"path": rel(ERROR_INSPECTION_PATH), "sha256": sha256_file(ERROR_INSPECTION_PATH)},
    "model_improvement_floor": {"path": rel(MODEL_FLOOR_PATH), "sha256": sha256_file(MODEL_FLOOR_PATH)},
    "acceptance_criteria": acceptance_criteria,
    "best_baseline": best_baseline,
    "best_baseline_validation_mae": best_val_metrics["mae"],
    "best_baseline_test_mae": best_test_metrics["mae"],
    "blockers": high_fit_coverage_failures + embedding_manifest.get("blockers", []),
    "notes": [
        "Manual Phase 16 labels remain evaluation-only and are not used as model input features.",
        "Cosine baseline is production-eligible E5 evidence only when sentence-transformers loads intfloat/e5-base-v2 successfully.",
    ],
}
write_json(PHASE_REPORT_PATH, phase_report)
phase_report


{'phase_id': 'phase_17_baseline_evaluation_v2',
 'schema_version': 'baseline-evaluation-v2',
 'status': 'complete',
 'generated_at': '2026-06-02T04:48:49.553494+00:00',
 'source': {'pairs_v2': {'path': 'artifacts/pairs_v2.parquet',
   'row_count': 3600,
   'sha256': '0876d3353a220dc5fe1f93654b4d7ae85a19fe9bd4fd9c132e11d9f47958cd8a'},
  'jobs': {'path': 'legacy/dataset/indotech_job_cleaned.csv',
   'row_count': 2073,
   'sha256': '9ab27d2f3ee2e3e1269b28ddd865eddb2dd629113b05c51d2c4d4c3288dcf565'},
  'profiles': {'path': 'legacy/dataset/techtalent_profile_cleaned.csv',
   'row_count': 69929,
   'sha256': '79ec1cda8d3c7fef86566e02171085910ed0a1c725acfbb1f8cc4c04a5512494'}},
 'embedding_manifest': {'path': 'reports/phase_17_embedding_manifest.json',
  'sha256': '517558eabb701432daab54cdc83add13df203e601bdae3c3a964ad7c8b55ab72'},
 'baseline_metrics': {'path': 'reports/phase_17_baseline_metrics.json',
  'sha256': '2c7362ca174f8b72b8cbe5953d65fb426d30e8d17e2bbfb0ab48513ec0f97021'},
 'slice_me